# 02 — Fine-Tuning Mistral 7B with QLoRA
Train the invoice extraction model using QLoRA on Kaggle's T4 GPU.

In [ ]:
!pip install -q transformers peft bitsandbytes trl datasets accelerate wandb sentencepiece protobuf scipy

In [ ]:
import os
import sys
sys.path.insert(0, "..")
import wandb
from src.training.config import TrainingConfig
from src.training.train import train
wandb.login()

## Configure Training

In [ ]:
config = TrainingConfig(
    model_name="mistralai/Mistral-7B-v0.3",
    lora_r=64,
    lora_alpha=128,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_seq_length=2048,
)
print(f"Model: {config.model_name}")
print(f"LoRA rank: {config.lora_r}, alpha: {config.lora_alpha}")
print(f"Effective batch size: {config.effective_batch_size}")
print(f"Epochs: {config.num_train_epochs}")

## Train

In [ ]:
trainer = train(
    config=config,
    train_path="../data/train.jsonl",
    eval_path="../data/eval.jsonl",
    output_dir="../outputs/",
)
print("Training complete!")
print(f"Best checkpoint saved to ../outputs/final_adapter")

## Verify Adapter

In [ ]:
import os
adapter_path = "../outputs/final_adapter"
files = os.listdir(adapter_path)
print(f"Adapter files: {files}")

from src.evaluation.evaluate import load_finetuned_model
model, tokenizer = load_finetuned_model(config.model_name, adapter_path)

test_input = "Invoice #TEST-001\nFrom: Test Corp\nDate: 2024-06-15\nDue: 2024-07-15\nItem: Widget x1 @ $100 = $100\nTotal: $110 (tax $10)\nCurrency: USD"
prompt = f"### Instruction:\nExtract all invoice fields from the following invoice text as JSON.\n\n### Input:\n{test_input}\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
import torch
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Model output:")
print(response)